# 04C Policy Robustness and Tradeoffs

Python-first implementation for non-regression analyses and figures.

Notes:
- `data/q_df.csv` is treated as read-only.
- Balanced scenario only (`0.25` each for `lexdiv`, `sentcomp`, `sim`, `smog`).
- Any future `glmmTMB` beta regressions remain in R files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

SEED = 12345
B = 1000

q_df_path = Path("data/q_df.csv")
out_dir = Path("model_output/policy_robustness")
fig_dir = out_dir / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

q_df_stat_before = q_df_path.stat()


In [ ]:
usecols = ["discussion", "sorting policy", "value", "n", "feature", "policy", "replies", "pinned"]

df = pd.read_csv(
    q_df_path,
    usecols=usecols,
    dtype={
        "discussion": "string",
        "sorting policy": "category",
        "value": "float32",
        "n": "string",
        "feature": "category",
        "policy": "category",
        "replies": "category",
        "pinned": "int8",
    },
    low_memory=False,
)

required = ["discussion", "sorting policy", "value", "n", "feature", "policy", "replies", "pinned"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in q_df.csv: {missing}")

if not ((df["value"] >= -1).all() and (df["value"] <= 1).all()):
    raise ValueError("q_df value must be in [-1, 1]")

df["discussion"] = df["discussion"].astype(str)
df["sorting policy"] = df["sorting policy"].astype(str)
df["feature"] = df["feature"].astype(str)
df["n"] = df["n"].astype(str)

df = df[df["n"].isin(["10", "N"])].copy()

print("rows:", len(df))
print("policies:", df["sorting policy"].nunique())
print("features:", sorted(df["feature"].unique().tolist()))
print("n values:", sorted(df["n"].unique().tolist()))


In [ ]:
def rank_min_desc(values):
    # tie rule: shared best rank for ties
    return pd.Series(values).rank(method="min", ascending=False).to_numpy()

def bootstrap_policy_ranks(df_in, B=1000, seed=12345):
    rng = np.random.default_rng(seed)

    cells = []
    for (feature, n_val), d_cell in df_in.groupby(["feature", "n"], sort=False):
        pivot = d_cell.pivot_table(index="discussion", columns="sorting policy", values="value", aggfunc="mean")
        pivot = pivot.sort_index(axis=0).sort_index(axis=1)

        discussions = pivot.index.to_numpy()
        policies = pivot.columns.to_numpy()
        V = pivot.to_numpy(dtype=float)
        D, P = V.shape

        boot_means = np.empty((B, P), dtype=float)
        boot_ranks = np.empty((B, P), dtype=float)

        for b in range(B):
            sampled = rng.integers(0, D, size=D)
            counts = np.bincount(sampled, minlength=D).astype(float)
            means = counts @ V / D
            ranks = rank_min_desc(means)
            boot_means[b, :] = means
            boot_ranks[b, :] = ranks

        point_means = V.mean(axis=0)

        summary = pd.DataFrame({
            "feature": feature,
            "n": n_val,
            "sorting policy": policies,
            "mean_forum": point_means,
            "ci_lower": np.quantile(boot_means, 0.025, axis=0),
            "ci_upper": np.quantile(boot_means, 0.975, axis=0),
            "median_rank": np.median(boot_ranks, axis=0),
            "rank_iqr": np.quantile(boot_ranks, 0.75, axis=0) - np.quantile(boot_ranks, 0.25, axis=0),
            "p_rank_1": (boot_ranks == 1).mean(axis=0),
            "p_rank_top3": (boot_ranks <= 3).mean(axis=0),
        })

        heat_rows = []
        for j, pol in enumerate(policies):
            rank_counts = pd.Series(boot_ranks[:, j]).value_counts().sort_index()
            for rank_value, cnt in rank_counts.items():
                heat_rows.append({
                    "feature": feature,
                    "n": n_val,
                    "sorting policy": pol,
                    "rank": int(rank_value),
                    "prob": cnt / B,
                })

        cells.append((summary, pd.DataFrame(heat_rows)))

    summary_df = pd.concat([c[0] for c in cells], ignore_index=True)
    heat_df = pd.concat([c[1] for c in cells], ignore_index=True)
    return summary_df, heat_df

bootstrap_summary, bootstrap_heat = bootstrap_policy_ranks(df, B=B, seed=SEED)

bootstrap_summary.to_csv(out_dir / "bootstrap_forum_summary.csv", index=False)
bootstrap_heat.to_csv(out_dir / "bootstrap_rank_stability.csv", index=False)

print("bootstrap outputs written")


In [ ]:
# Figure 1: policy mean FORUM with 95% CI
features = sorted(bootstrap_summary["feature"].unique())
n_levels = ["10", "N"]

fig, axes = plt.subplots(len(features), len(n_levels), figsize=(14, 10), sharex=False)
if len(features) == 1:
    axes = np.array([axes])

for i, feat in enumerate(features):
    for j, n_val in enumerate(n_levels):
        ax = axes[i, j]
        s = bootstrap_summary[(bootstrap_summary["feature"] == feat) & (bootstrap_summary["n"] == n_val)].copy()
        s = s.sort_values("mean_forum")
        y = np.arange(len(s))
        ax.errorbar(
            s["mean_forum"],
            y,
            xerr=[s["mean_forum"] - s["ci_lower"], s["ci_upper"] - s["mean_forum"]],
            fmt="o",
            markersize=3,
            linewidth=1,
            color="#1f77b4",
            ecolor="#4c78a8",
            capsize=1.5,
        )
        ax.set_title(f"{feat} | n={n_val}")
        ax.set_yticks(y)
        ax.set_yticklabels(s["sorting policy"], fontsize=7)
        ax.set_xlabel("Mean FORUM")

plt.tight_layout()
plt.savefig(fig_dir / "policy_mean_forum_bootstrap_ci.png", dpi=160)
plt.close()

# Figure 2: rank stability heatmap
fig, axes = plt.subplots(len(features), len(n_levels), figsize=(14, 10), sharex=False, sharey=False)
if len(features) == 1:
    axes = np.array([axes])

for i, feat in enumerate(features):
    for j, n_val in enumerate(n_levels):
        ax = axes[i, j]
        h = bootstrap_heat[(bootstrap_heat["feature"] == feat) & (bootstrap_heat["n"] == n_val)].copy()
        mat = h.pivot(index="sorting policy", columns="rank", values="prob").fillna(0)
        mat = mat.loc[sorted(mat.index)]
        sns.heatmap(mat, ax=ax, cmap="Blues", cbar=(i == 0 and j == len(n_levels)-1))
        ax.set_title(f"{feat} | n={n_val}")
        ax.set_xlabel("Rank")
        ax.set_ylabel("Sorting policy")

plt.tight_layout()
plt.savefig(fig_dir / "rank_stability_heatmap.png", dpi=160)
plt.close()


In [ ]:
# Metric triangulation from existing FORUM depth anchors (n=10 and n=N)
pair = (
    df[df["n"].isin(["10", "N"])]
    .pivot_table(index=["feature", "discussion", "sorting policy"], columns="n", values="value", aggfunc="mean")
    .reset_index()
)
pair = pair.dropna(subset=["10", "N"]).copy()

pair["g10"] = ((pair["10"] + 1) / 2).clip(0, 1)
pair["gN"] = ((pair["N"] + 1) / 2).clip(0, 1)

pair["FORUM_10"] = pair["10"]
pair["FORUM_N"] = pair["N"]
pair["nDCG_10"] = (2 ** pair["g10"] - 1)
pair["AUC_10"] = pair["g10"]
pair["nDCG_N"] = (((2 ** pair["g10"] - 1) / np.log2(2)) + ((2 ** pair["gN"] - 1) / np.log2(3))) / (((2 ** 1 - 1) / np.log2(2)) + ((2 ** 1 - 1) / np.log2(3)))
pair["AUC_N"] = (pair["g10"] + pair["gN"]) / 2

scores_10 = pair.groupby(["feature", "sorting policy"], as_index=False).agg(FORUM=("FORUM_10", "mean"), nDCG=("nDCG_10", "mean"), AUC=("AUC_10", "mean"))
scores_10["n"] = "10"

scores_N = pair.groupby(["feature", "sorting policy"], as_index=False).agg(FORUM=("FORUM_N", "mean"), nDCG=("nDCG_N", "mean"), AUC=("AUC_N", "mean"))
scores_N["n"] = "N"

tri_scores = pd.concat([scores_10, scores_N], ignore_index=True)[["feature", "n", "sorting policy", "FORUM", "nDCG", "AUC"]]
tri_scores.to_csv(out_dir / "metric_triangulation_scores.csv", index=False)

corr_rows = []
for (feat, n_val), d in tri_scores.groupby(["feature", "n"]):
    corr_rows.append({
        "feature": feat,
        "n": n_val,
        "spearman_forum_ndcg": d["FORUM"].corr(d["nDCG"], method="spearman"),
        "spearman_forum_auc": d["FORUM"].corr(d["AUC"], method="spearman"),
    })

tri_corr = pd.DataFrame(corr_rows)
tri_corr.to_csv(out_dir / "metric_rank_correlations.csv", index=False)

print("triangulation outputs written")


In [ ]:
# Figure 3: metric rank comparison
rank_df = tri_scores.melt(
    id_vars=["feature", "n", "sorting policy"],
    value_vars=["FORUM", "nDCG", "AUC"],
    var_name="metric",
    value_name="score",
)
rank_df["rank"] = rank_df.groupby(["feature", "n", "metric"])["score"].rank(method="min", ascending=False)

features = sorted(rank_df["feature"].unique())
n_levels = ["10", "N"]
metrics_order = ["FORUM", "nDCG", "AUC"]
metric_pos = {m: i for i, m in enumerate(metrics_order)}

fig, axes = plt.subplots(len(features), len(n_levels), figsize=(14, 10), sharey=True)
if len(features) == 1:
    axes = np.array([axes])

for i, feat in enumerate(features):
    for j, n_val in enumerate(n_levels):
        ax = axes[i, j]
        d = rank_df[(rank_df["feature"] == feat) & (rank_df["n"] == n_val)].copy()
        for pol, dp in d.groupby("sorting policy"):
            dp = dp.set_index("metric").reindex(metrics_order).reset_index()
            x = [metric_pos[m] for m in dp["metric"]]
            y = dp["rank"].to_numpy()
            ax.plot(x, y, alpha=0.5, linewidth=1)
            ax.scatter(x, y, s=8)
        ax.set_title(f"{feat} | n={n_val}")
        ax.set_xticks(list(metric_pos.values()))
        ax.set_xticklabels(metrics_order)
        ax.set_xlabel("Metric")
        ax.invert_yaxis()
        if j == 0:
            ax.set_ylabel("Rank (1 = best)")

plt.tight_layout()
plt.savefig(fig_dir / "metric_rank_comparison.png", dpi=160)
plt.close()


In [ ]:
# Multi-objective tradeoff: balanced scenario only
obj = (
    df.groupby(["n", "sorting policy", "feature"], as_index=False)["value"].mean()
    .pivot(index=["n", "sorting policy"], columns="feature", values="value")
    .reset_index()
)

for c in ["lexdiv", "sentcomp", "sim", "smog"]:
    if c not in obj.columns:
        raise ValueError(f"Missing objective column: {c}")


def pareto_flags(mat):
    # True means pareto efficient (non-dominated)
    n = mat.shape[0]
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if np.all(mat[j] >= mat[i]) and np.any(mat[j] > mat[i]):
                dominated[i] = True
                break
    return ~dominated

pareto_parts = []
for n_val, d in obj.groupby("n", sort=False):
    d = d.copy()
    mat = d[["lexdiv", "sentcomp", "sim", "smog"]].to_numpy()
    d["pareto_efficient"] = pareto_flags(mat)
    pareto_parts.append(d)

pareto_df = pd.concat(pareto_parts, ignore_index=True)
pareto_df["dominated"] = ~pareto_df["pareto_efficient"]

balanced_df = pareto_df.copy()
balanced_df["w_lexdiv"] = 0.25
balanced_df["w_sentcomp"] = 0.25
balanced_df["w_sim"] = 0.25
balanced_df["w_smog"] = 0.25
balanced_df["balanced_score"] = 0.25 * balanced_df["lexdiv"] + 0.25 * balanced_df["sentcomp"] + 0.25 * balanced_df["sim"] + 0.25 * balanced_df["smog"]

winners = balanced_df.loc[balanced_df.groupby("n")["balanced_score"].transform("max") == balanced_df["balanced_score"], ["n", "sorting policy", "balanced_score", "pareto_efficient"]]

pareto_df.to_csv(out_dir / "pareto_frontier.csv", index=False)
balanced_df.to_csv(out_dir / "balanced_scenario_scores.csv", index=False)
winners.to_csv(out_dir / "balanced_winner_by_n.csv", index=False)

print("multi-objective outputs written")


In [ ]:
# Figure 4a: pareto highlight (sim vs smog)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False, sharey=False)
n_levels = ["10", "N"]

for j, n_val in enumerate(n_levels):
    ax = axes[j]
    d = balanced_df[balanced_df["n"] == n_val].copy()
    sc = ax.scatter(
        d["sim"],
        d["smog"],
        c=d["balanced_score"],
        cmap="OrRd",
        s=np.where(d["pareto_efficient"], 80, 35),
        marker="^",
        edgecolors="k",
        linewidths=0.3,
        alpha=0.9,
    )
    ax.set_title(f"n={n_val}")
    ax.set_xlabel("Mean FORUM (sim)")
    ax.set_ylabel("Mean FORUM (smog)")

cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.9)
cbar.set_label("Balanced score")
plt.tight_layout()
plt.savefig(fig_dir / "pareto_highlight_plot.png", dpi=160)
plt.close()

# Figure 4b: balanced winner chart
fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharex=False)
for j, n_val in enumerate(n_levels):
    ax = axes[j]
    d = balanced_df[balanced_df["n"] == n_val].copy().sort_values("balanced_score")
    winner_score = d["balanced_score"].max()
    colors = np.where(np.isclose(d["balanced_score"], winner_score), "#238b45", "#bdbdbd")
    ax.barh(d["sorting policy"], d["balanced_score"], color=colors)
    ax.set_title(f"n={n_val}")
    ax.set_xlabel("Balanced score")
    ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig(fig_dir / "balanced_winner_chart.png", dpi=160)
plt.close()


In [ ]:
# Acceptance checks
expected_files = [
    "bootstrap_forum_summary.csv",
    "bootstrap_rank_stability.csv",
    "metric_triangulation_scores.csv",
    "metric_rank_correlations.csv",
    "pareto_frontier.csv",
    "balanced_scenario_scores.csv",
    "balanced_winner_by_n.csv",
]

for name in expected_files:
    p = out_dir / name
    if not p.exists():
        raise FileNotFoundError(f"Missing output file: {p}")
    if len(pd.read_csv(p)) == 0:
        raise ValueError(f"Empty output file: {p}")

q_df_stat_after = q_df_path.stat()
if q_df_stat_before.st_size != q_df_stat_after.st_size:
    raise RuntimeError("q_df.csv size changed unexpectedly")
if q_df_stat_before.st_mtime != q_df_stat_after.st_mtime:
    raise RuntimeError("q_df.csv mtime changed unexpectedly")

print("All acceptance checks passed.")
